<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 (AI): Embeddings, Vector Search & Chunking

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. See **why keyword search fails** when the words differ but the meaning doesn't
2. Turn text into **embeddings** with a free local model — no API key needed
3. Measure closeness with **cosine similarity**
4. **See the space** — squash 384 dimensions into a 2-D picture and watch topics cluster
5. Call **OpenAI embeddings** two ways (LiteLLM and the native SDK) — 384 vs 1536 dimensions
6. **Chunk** a document — watch a naive splitter destroy a fact, then fix it properly
7. **Store and search** vectors in a **Chroma** vector database
8. Snap an LLM on the end and get **RAG** in five lines

> **Sections 2–4 need no API key at all.** Only the LiteLLM comparison (§5) and the RAG teaser (§9) do.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q chromadb sentence-transformers scikit-learn litellm langchain-text-splitters matplotlib

In [ ]:
import os
from getpass import getpass
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

print("Imports ready")

---

## 2. The Problem — keyword search doesn't understand meaning

Every search you've written so far matches **letters**: `Ctrl+F`, `WHERE body LIKE '%refund%'`, `grep`.

It asks *"do these characters appear?"* — never *"is this about the same thing?"*

In [ ]:
# Two texts that mean the same thing to a human
question = "How do I cancel my order?"
document = "Returns and refunds policy"

# Keyword search asks only one thing: which words do they share?
q_words = set(question.lower().strip("?").split())
d_words = set(document.lower().split())

print("Words in common:", q_words & d_words)

**Zero words in common** — so keyword search scores the single most relevant page in the manual as a perfect miss.

The failure runs both ways:

* **Misses** — *cancel / return / refund / send back* are one idea to a human and four strings to `LIKE`
* **False hits** — search `apple` and get the fruit, the company, and someone's surname

> 💡 **The idea that fixes it:** if we could turn text into **numbers**, where similar meaning lands on **nearby numbers**, then searching by meaning becomes plain geometry — just measure which points are close.

---

## 3. Load the Embedding Model

An **embedding** is a list of numbers that represents the **meaning** of a piece of text.

`all-MiniLM-L6-v2` is a small, fast, **free** model that runs right here on the Colab CPU.
The first run downloads it (about a minute); after that it's instant.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded")

In [ ]:
texts = [
    "Machine learning is a branch of AI",
    "Deep learning uses neural networks",
    "I love cooking Indian food",
    "Cricket is popular in India"
]

embeddings = model.encode(texts)

print(f"Shape: {embeddings.shape}")   # (4, 384) -> 4 texts, 384 numbers each

`(4, 384)` — four texts in, four vectors out, and **each one is exactly 384 numbers long**.

Try it yourself: change one of the sentences to a whole paragraph and re-run. The shape stays `(4, 384)`.
Three words or three paragraphs — **always the same length**. That fixed size is what makes comparison possible at all.

In [ ]:
# What does one actually look like?
print("First 8 numbers of the first embedding:")
print(embeddings[0][:8])

Think of a world map: two numbers (latitude, longitude) place any city, and cities near in the numbers are
near in reality. An embedding is that idea with **384 axes** instead of 2 — too many to draw, identical logic.

> 💡 **The words are gone.** From here on, nothing compares text to text. You are comparing 384 numbers to 384 numbers.

---

## 4. Semantic Similarity

We have vectors. We need a **single number** for "how close are these two?"

**Cosine similarity** measures the **angle** between two vectors and ignores their length:
**1.0** = same direction (same meaning), **0** = unrelated.

Why the angle and not the distance? Length mostly tracks *how long or emphatic* the text is.
**Direction carries the meaning.**

In [ ]:
similarity_matrix = cosine_similarity(embeddings)

# Compare: similar topics vs different topics
print(f"ML vs DL (similar topics):    {similarity_matrix[0][1]:.3f}")
print(f"ML vs Cooking (different):    {similarity_matrix[0][2]:.3f}")
print(f"ML vs Cricket (different):    {similarity_matrix[0][3]:.3f}")

Read that carefully: **"Machine learning is a branch of AI"** and **"Deep learning uses neural networks"**
share almost no words — and they still score far higher than the cooking and cricket sentences.

The meaning survived the trip into numbers.

⚠️ Those numbers are **model-specific**. Never hard-code `similarity > 0.8 means relevant` — it won't
transfer to another model or another dataset. What *is* reliable is the **ranking**, which is why every
retrieval API asks for **top-k** rather than "everything above a threshold".

In [ ]:
# The whole matrix - every sentence against every other one
import pandas as pd

pd.DataFrame(similarity_matrix.round(3), index=texts, columns=["ML", "DL", "Cooking", "Cricket"])

The diagonal is 1.000 — every sentence is identical to itself. Look for the bright pair off the diagonal.

---

## 5. See the Space — visualising embeddings

A table of numbers is convincing. A **picture** is unforgettable.

Nobody can picture 384 dimensions — but we can **squash** them down to 2 and look at the shadow.
That's what **PCA** does: it finds the two directions that carry the most variation and throws the
rest away.

Let's use more sentences so there's something to see: three about **AI**, three about **food**,
three about **sport** — and no labels given to the model. Just the raw text.

In [ ]:
many_texts = [
    # --- AI ---
    "Machine learning is a branch of AI",
    "Deep learning uses neural networks",
    "Neural networks learn from training data",
    # --- food ---
    "I love cooking Indian food",
    "This biryani recipe needs saffron",
    "The restaurant serves excellent pasta",
    # --- sport ---
    "Cricket is popular in India",
    "He scored a century in the match",
    "The football team won the league",
]

many_vectors = model.encode(many_texts)
print(many_vectors.shape)     # (9, 384)

In [ ]:
from sklearn.decomposition import PCA

# Squash 384 dimensions down to 2 so we can actually look at them
coords = PCA(n_components=2).fit_transform(many_vectors)

print(many_vectors.shape, "->", coords.shape)

In [ ]:
import matplotlib.pyplot as plt

colors = ["#6366F1"] * 3 + ["#0891B2"] * 3 + ["#B91C1C"] * 3   # AI / food / sport

plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=160)

# label each point with its sentence
for (x, y), label in zip(coords, many_texts):
    plt.annotate(label, (x, y), fontsize=9, xytext=(8, 5), textcoords="offset points")

plt.title("384 dimensions squashed into 2 — the topics separate on their own")
plt.axis("off")
plt.tight_layout()
plt.show()

**Three clusters, and nobody told the model what the topics were.** We never passed a label, a
category or a tag — only nine sentences. The grouping is what the *meaning* looks like when you
plot it.

> 💡 **This is the whole day in one picture.** Search "by meaning" is just: put the question on this
> map, and return whatever is nearest.

⚠️ **Be honest about what you're looking at.** PCA throws away 382 of the 384 dimensions, so this
picture is a **shadow**, not the truth. Two points that look close here might be further apart in
the real space. Use the picture to build intuition — use `cosine_similarity` to make decisions.

🎨 **Try it:** add a sentence of your own to `many_texts`, re-run the three cells, and see where it lands.
Try one that *bridges* two clusters — `"AI is used to analyse cricket matches"` — and watch it fall
somewhere in between.

### Want to explore a much bigger space?

The **[TensorFlow Embedding Projector](https://projector.tensorflow.org/)** does exactly what we just
did, but interactively and at scale — 10,000 Word2Vec words in 3-D, which you can rotate, search and
zoom. Click any word to see its nearest neighbours.

**Two minutes well spent in class:**
1. Open <https://projector.tensorflow.org/> (it loads Word2Vec 10K by default — nothing to install)
2. Search for **`india`** in the box on the right → look at the nearest neighbours it lists
3. Switch the projection from **PCA** to **t-SNE** and watch the clusters reorganise
4. Try **`king`**, then **`teacher`**, then a word from your own field

*(Other tools worth knowing: [Nomic Atlas](https://atlas.nomic.ai/) for large text/image sets, and
[Apple's Embedding Atlas](https://github.com/apple/embedding-atlas) for exploring millions of points locally.)*

---

## 6. OpenAI Embeddings — two ways to call them

So far everything has run on the **local** model. The hosted models are usually **more accurate**, and
they cost almost nothing: `text-embedding-3-small` is about **$0.02 per million tokens**, so embedding
an entire novel costs under a rupee.

**This section needs an OpenAI key.** Skip it if you don't have one — everything after it still works
on the local model.

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass("Enter your OpenAI API Key: ")

### Way 1 — LiteLLM (one function, any provider)

In [ ]:
from litellm import embedding

response = embedding(model="text-embedding-3-small", input=texts)

print(f"Model used:           {response.model}")
print(f"Number of embeddings: {len(response.data)}")
print(f"OpenAI dimensions:    {len(response.data[0]['embedding'])}")
print(f"Local dimensions:     {embeddings.shape[1]}")
print(f"\nFirst 5 values: {response.data[0]['embedding'][:5]}")

### Way 2 — the native OpenAI SDK

Same vectors, different door. Use this when you're already using the OpenAI client elsewhere.

In [ ]:
from openai import OpenAI

client = OpenAI()

r = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,                       # a list -> one call, many vectors
)
openai_vectors = [d.embedding for d in r.data]

print(f"{len(openai_vectors)} vectors, {len(openai_vectors[0])} numbers each")
print(f"Tokens billed: {r.usage.total_tokens}")

In [ ]:
# Ask for a SHORTER vector - cheaper to store, faster to search, slightly less accurate
short = client.embeddings.create(
    model="text-embedding-3-small",
    input="How do I cancel my order?",
    dimensions=256,                    # <-- try 512, 1024, 1536
).data[0].embedding

print("Shrunk to:", len(short), "numbers")

**1536 numbers vs 384.** Same four sentences, two completely different coordinate systems.

> ⚠️ **The rule you must not break:** use the **same** embedding model for your documents **and** your
> queries. Two models = two different spaces. Mixing them returns nonsense — and it does **not** raise an error.
> Change your embedding model later and you must **re-embed everything**.

### Running the whole notebook on OpenAI embeddings

Everything below uses `model.encode(...)`. To switch the entire pipeline to OpenAI, you only need to
replace that one call — nothing else changes. **Uncomment and run this to swap:**

In [ ]:
# ---- OPTIONAL: use OpenAI embeddings everywhere below this cell ----
# import numpy as np
#
# class OpenAIEmbedder:
#     def encode(self, texts):
#         if isinstance(texts, str):
#             texts = [texts]
#         r = client.embeddings.create(model="text-embedding-3-small", input=list(texts))
#         return np.array([d.embedding for d in r.data])
#
# model = OpenAIEmbedder()      # same .encode() interface, 1536 dimensions now
# print("Switched to OpenAI embeddings")

🧑‍🏫 That's the point worth remembering: **the embedding model is a swappable part.** Chunking,
storage, search and RAG all stay exactly the same — only the vectors underneath change. Just
remember rule 1: if you swap it, **re-index everything.**

---

## 7. A Knowledge Base, and How to Cut It Up

Here is the document we'll search — a company profile with a lot of specific facts in it.

In [ ]:
company_info = """
TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma. The company is headquartered in Bhubaneswar, Odisha, with additional offices in Bangalore and Hyderabad. TechSolutions specializes in AI/ML solutions, cloud services, and mobile applications, and has grown to over 250 employees. The company achieved 50 crores in annual revenue in 2024.

Priya Sharma is the CEO and co-founder. She graduated from IIT Delhi and completed her MBA from Stanford University. She previously worked as Senior Director at Infosys and won the Women in Tech Leader award in 2022. Rahul Verma is the CTO and co-founder. He graduated from BITS Pilani and previously worked as Tech Lead at Google India. His expertise is in Machine Learning and Cloud Architecture. Ananya Patel is the VP of Engineering and manages a team of 100+ engineers.

TechSolutions offers three main products. CloudAssist Pro is the flagship enterprise cloud management platform, costing 50,000 per month, and includes auto-scaling, real-time monitoring, cost optimization, and 24/7 support. SmartHR is an AI-powered HR management system at 25,000 per month that handles recruitment automation, payroll processing, performance tracking, and employee analytics. DataViz Analytics is a business intelligence platform at 15,000 per month with real-time dashboards, custom reports, and predictive analytics.

Work hours at TechSolutions are 9 AM to 6 PM, Monday to Friday, following a hybrid model with 3 days in office and 2 days remote. Employees receive 24 paid leaves and 10 sick leaves per year. Maternity leave is 26 weeks and paternity leave is 2 weeks. Probation period is 6 months for all new employees, with a notice period of 2 months for permanent employees and 1 month during probation.

Employee benefits include health insurance coverage of 5 lakh for employees and their families, covering hospitalization, OPD, dental, and vision. The learning budget is 50,000 per year per employee for courses, certifications, and conferences. Performance bonuses can be up to 20% of annual salary. Major clients include HDFC Bank, Tata Motors, Reliance Industries, and ICICI Bank, with a 95% customer satisfaction rate across 200+ completed projects.
"""

print(f"Document length: {len(company_info)} characters")

**Why not just embed the whole thing?** Because an embedding is **one point in space**, and this document
talks about founders, prices, leave policy and clients all at once. One point cannot represent forty ideas —
the meaning averages into mush, and it becomes a good match for nothing.

So we **chunk** it. But *how* you cut matters enormously.

### First, the wrong way — cut every N characters

The obvious approach: just slice every 200 characters. Watch what it does.

In [ ]:
def naive_chunks(text, size):
    """The obvious approach - and a trap. Slice every `size` characters."""
    return [text[i:i + size] for i in range(0, len(text), size)]


for i, c in enumerate(naive_chunks(company_info, 200)[2:4], start=2):
    print(f"--- chunk {i} ---")
    print(repr(c))
    print()

Look at where the cut lands:

```
chunk 2:  "... Rahul Verma is the CT"
chunk 3:  "O and co-founder. He graduated from BITS Pilani ..."
```

It sliced **CTO straight down the middle.** Now ask *"who is the CTO?"* — the word **CTO does not exist
in any chunk**. One ends in "CT", the next begins with a stray "O". The fact is gone, and no model on
earth can recover it.

> 💡 **Bad chunking beats a good model.** You can pay for the most expensive model available and still
> lose to someone with a cheap model and sensible boundaries.

### Now the right way — `RecursiveCharacterTextSplitter`

This is why you don't hand-roll a chunker. The recursive splitter tries to break on a **paragraph** first,
then a **sentence**, then a **word** — only ever cutting mid-word as a last resort. It also repeats a little
text between chunks (`chunk_overlap`) so a fact split at a boundary still survives whole in its neighbour.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_text(company_info)

print(f"Split into {len(chunks)} chunks\n")
print("--- chunk 4 ---")
print(chunks[4])

Every chunk now ends on a **whole word**, and "Rahul Verma is the CTO and co-founder" stays intact.

**Change `chunk_size` and `chunk_overlap` above and re-run** to feel the trade-off:
* ~200–500 tokens (≈ 800–2,000 characters) is the usual sweet spot
* 10–20% overlap is the usual insurance
* Dense FAQ text → smaller. Flowing prose → bigger.

The honest answer to "what size?" is **measure it**: take 20 real questions, try two settings, and count
how often the right chunk lands in the top 3.

In [ ]:
# Look at them all
for i, chunk in enumerate(chunks):
    print(f"[{i}] ({len(chunk)} chars) {chunk[:70]}...")

---

## 8. Store in a Chroma Vector Database

We could keep these vectors in a Python list and compare them with a loop — and honestly, for a few
thousand chunks that works fine. A **vector database** earns its place when you need:

| | With a plain list | With a vector DB |
|---|---|---|
| **Speed** | compare against every vector | an ANN index — approximate, ~100× faster |
| **Persistence** | restart = re-embed = re-pay | stored on disk |
| **Filtering** | write your own loop | `where={"year": 2026}` before searching |
| **Updates** | rebuild the array | add / update / delete by `id` |

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="techsolutions")

print("Collection ready")

In [ ]:
chunk_embeddings = model.encode(chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)

print(f"Added {collection.count()} chunks to collection")

Three things went in together: your own **`ids`** (so you can update or delete later), the **`embeddings`**,
and the original **`documents`** so you get the text back rather than just a row number.

You can also attach **`metadatas`** — `source`, `page`, `date` — which lets you *filter before searching*
and *cite the real source* afterwards. Citations come from your metadata, never from the model's memory.

---

## 9. Semantic Search

Two phases, and they are **not** the same thing:

**Indexing** (offline, once, whenever documents change) — `documents → chunk → embed → store`
**Querying** (live, on every request) — `query → embed with the SAME model → search → top-k`

In [ ]:
def search(query, n_results=3):
    """Search for relevant documents."""
    query_embedding = model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    return results

In [ ]:
# Change the query and re-run
query = "Who founded the company?"          # <-- change this

results = search(query, n_results=2)

print(f"Query: {query}\n")
for doc in results['documents'][0]:
    print(f"  -> {doc[:90]}...")

**Try these one at a time**, changing `query` above:

* `"What are the product prices?"`
* `"How many leaves do employees get?"`
* `"Who are the big customers?"` — note: the document never says "customers", it says **clients**
* `"work from home policy"` — the document never uses the word "remote work" either

That last pair is the whole point: **the words don't match, and the right chunk still comes back.**

> ⚠️ **Now break it on purpose.** Ask something the document never mentions — `"what is the hostel fee?"`
> — and watch it **still confidently return chunks**. Similarity search always returns its top-k; it has no
> concept of "nothing here is relevant." Fixing that is a big part of Day 4.

In [ ]:
# Chroma also gives you the distances - LOWER means closer
results = search("How many leaves do employees get?", n_results=3)

for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"{dist:.3f}  {doc[:70]}...")

⚠️ Two things that trip everyone up:

1. Chroma returns **`distances`, not similarities** — **lower is better**.
2. Results are **lists of lists** (`results['documents'][0]`) because `query` accepts a *batch* of queries.

---

## 10. RAG in Five Lines

You can now find the right chunks. You already have a model that writes. Snap them together.

In [ ]:
from litellm import completion

question = "Who is the CTO and what is his background?"

# 1) RETRIEVAL - today's entire lesson
context = "\n\n".join(search(question, n_results=3)['documents'][0])

# 2) AUGMENTED - put the chunks in the prompt
# 3) GENERATION - an ordinary chat call
answer = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content":
            "Answer using ONLY the context below. If it is not there, say you don't know.\n\n" + context},
        {"role": "user", "content": question},
    ],
    temperature=0,
)

print(answer.choices[0].message.content)

**That is RAG** — **R**etrieval **A**ugmented **G**eneration:

| Letter | What it is | Where it came from |
|---|---|---|
| **R**etrieval | find the right chunks | everything you built today |
| **A**ugmented | put them in the prompt | one `join()` |
| **G**eneration | the model answers | an ordinary chat call |

Note `temperature=0`. For RAG you want the model **reading**, not inventing — so you turn the creativity down.

> 💡 **Why this kills hallucination:** before, the model **recalled** from training and made things up when
> it didn't know. Here it **reads** text you handed it a millisecond ago. It stops being a memory and becomes a reader.

Five lines gets a demo. A **product** needs the hard parts — making it genuinely say "I don't know", real
citations, handling retrieval that returns junk, a proper chat interface, and measuring whether any of it
works. That's Day 4.

---

## 11. Exercises

Fill in the blanks (`___`) and run each cell.

### Q1: Rank sentences by meaning

Embed four sentences of your own and find which two are closest.

In [ ]:
# Hint: model.encode(list_of_texts) makes the vectors.
#       cosine_similarity(vectors) gives you the full matrix.

my_texts = [
    "The library is open until 10pm",
    "You can study in the library at night",
    "Biryani is best with raita",
    "___",                                  # add a fourth sentence of your own
]

my_vectors = model.___(my_texts)
my_matrix = ___(my_vectors)

print("Sentence 0 vs 1 (should be HIGH):", round(my_matrix[0][1], 3))
print("Sentence 0 vs 2 (should be LOW): ", round(my_matrix[0][2], 3))

### Q2: Break the chunker, then fix it

Show the naive splitter cutting a word in half, then fix it with the recursive splitter.

In [ ]:
# Hint: naive_chunks(text, size) is the bad one - a size near 45 cuts "hours" in half.
#       RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10) is the good one.

FACT = "The exam hall opens at 9am. The paper is 3 hours long. Calculators are not allowed."

broken = naive_chunks(FACT, ___)             # watch it slice a word down the middle
print("BROKEN:", broken)

good_splitter = RecursiveCharacterTextSplitter(chunk_size=___, chunk_overlap=___)
print("FIXED: ", good_splitter.split_text(FACT))   # every chunk ends on a whole word

### Q3: Index your own text and search it

Add three facts of your own to a new collection, then search it by meaning.

In [ ]:
# Hint: collection.add(ids=, embeddings=, documents=)  ·  .query(query_embeddings=, n_results=)

my_docs = [
    "The canteen serves lunch from 12pm to 2pm",
    "Assignments must be submitted on the portal by Friday",
    "The bus to campus leaves every 20 minutes",
]

my_collection = chroma_client.get_or_create_collection(name="exercise_q3")

my_collection.add(
    ids=["d0", "d1", "d2"],
    embeddings=model.encode(my_docs).___(),      # Chroma wants a plain list, not a numpy array
    documents=___,
)

# Ask something that shares NO words with the right answer
res = my_collection.query(
    query_embeddings=model.encode(["when can I eat?"]).tolist(),
    n_results=___,
)

print(res['documents'][0])

### Q4: Your own five-line RAG

Retrieve from the TechSolutions knowledge base, then answer **only** from what you retrieved.

In [ ]:
# Hint: search(question, n_results=3)['documents'][0] is a list of chunk texts.
#       Join them with "\n\n".

my_question = "What is the leave policy?"

my_context = "\n\n".join(___(my_question, n_results=3)['documents'][0])

reply = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer using ONLY this context:\n\n" + my_context},
        {"role": "user", "content": ___},
    ],
    temperature=___,                            # we want reading, not inventing
)

print(reply.choices[0].message.content)

---

### ✅ Recap

| Idea | The one-liner |
|---|---|
| **Embedding** | text → a fixed-length list of numbers that carries meaning |
| **PCA to 2-D** | a *shadow* of the real space — great for intuition, not for decisions |
| **`model.encode()`** | `all-MiniLM-L6-v2` gives 384 numbers, free and local |
| **Same model both sides** | documents and queries must share a coordinate space — or you get silent nonsense |
| **Cosine similarity** | the angle between two vectors: 1 = same meaning, 0 = unrelated |
| **Chunking** | a naive slice cuts words in half — `RecursiveCharacterTextSplitter` breaks on real boundaries |
| **Overlap** | repeat a little text so a fact split at a boundary survives whole |
| **Chroma** | `add(ids, embeddings, documents)` then `query(...)` — and `distances`, where lower is better |
| **Top-k** | search always returns *something* — it never knows when nothing is relevant |
| **RAG** | retrieve the chunks → put them in the prompt → let the model read instead of recall |

**Next — Day 4:** RAG end-to-end — a real chat interface, grounding, citations, "I don't know", and what to do when retrieval fails.